# Notebook 17 - Video Understanding, Frame Sampling, and Temporal Reasoning

This notebook adds the video layer with an explicit constraint: stay honest about compute, so use sampled frames and bounded clips instead of unconstrained full-video fantasy workflows.


## What you will learn

- how to turn clips into sampled-frame workloads
- how to choose a temporal sampling policy
- how to package short-video prompts for multimodal models


In [ ]:
!pip install -q "transformers>=4.57.0" accelerate torch numpy pandas matplotlib decord opencv-python imageio
print("Installed notebook dependencies.")


In [ ]:
import json
import math
import random
import statistics
from collections import Counter, defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 140)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    CACHE_DIR = Path("/content/drive/MyDrive/models")
    ARTIFACT_ROOT = Path("/content/drive/MyDrive/castalia-multimodal")
except Exception:
    CACHE_DIR = Path("./models")
    ARTIFACT_ROOT = Path("./artifacts")

CACHE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
NOTEBOOK_ROOT = ARTIFACT_ROOT / "17_video_understanding_frame_sampling_and_temporal_reasoning"
NOTEBOOK_ROOT.mkdir(parents=True, exist_ok=True)

random.seed(7)
np.random.seed(7)



print("Cache dir:", CACHE_DIR)
print("Artifact root:", NOTEBOOK_ROOT.resolve())


## Step 1: Build a frame sampling schedule

Sampling policy is one of the biggest practical levers in short-video systems.


In [ ]:
def build_sampling_schedule(duration_seconds, fps=1):
    return [round(second, 2) for second in np.arange(0, duration_seconds, 1 / fps)]

schedule = build_sampling_schedule(duration_seconds=12, fps=1)
print(schedule)
print("Sampled frame count:", len(schedule))


## Step 2: Represent the sampled clip

A clean representation keeps the frame span tied to later answers.


In [ ]:
clip = pd.DataFrame([
    {"frame_id": idx, "time_s": second, "note": "placeholder frame"}
    for idx, second in enumerate(schedule[:8])
])
clip


## Step 3: Package a video prompt

Qwen2.5-VL accepts video blocks alongside text, but you still need to define the task and output contract clearly.


In [ ]:
video_prompt = [
    {
        "role": "user",
        "content": [
            {"type": "video", "path": "demo.mp4", "label": "screen recording"},
            {"type": "text", "text": "Summarize the key failure in the clip and cite the relevant frame window."},
        ],
    }
]
print(json.dumps(video_prompt, indent=2))


## Exercises and extensions

1. Compare 1 fps and 2 fps sampling on the same clip and compute the cost multiplier.
1. Add a heuristic that increases fps only around high-motion windows.
